# RecordDiff: Semi-Synthetic Bridge (mechanism recovery)

**Why this phase.** On real data the value-on-recording mechanism is non-identifiable, so we can never check whether the model recovered it. Here we inject a known mechanism into realistic data, so
`|b̂ − b_true|` is measurable, which validates the full model machinery (inferred latent `z`, 3-phase training, generation) before we trust it on real fidelity claims where ground truth is gone.

**Design.**
1. Generate complete synthetic values `x` calibrated to your real per-variable marginals (means/stds from the cached cohort), coupled to a latent AR severity trajectory.
2. Define a severity score `s = severity(x)`, which is a fixed, sample-independent function of the values.
3. Inject a measurement policy `P(m_v = 1) = σ(a_v + b_v · s)` where `b_v` follows the triggered/acuity/protocol taxonomy (scaled by `b_scale`), and each intercept `a_v` is calibrated so the variable's marginal missing rate matches the real cohort. This is a genuine MNAR mechanism: measurement depends on the (partly unobserved) true clinical state.
4. Train a fresh RecordDiff on the injected `(m, y = m ⊙ x, c)`.
5. Recover `b̂` by re-estimating the same per-variable logistic on the model's own generated samples (using severity of the model's complete generated values). If the model learned the mechanism, `b̂_gen ≈ b_true`.

**Controls.** `b_scale = 0` (null) must recover `b̂ ≈ 0`, which is the principled negative control real data can't provide. `b̂_inj` (estimator on the injected data) confirms the estimator itself is unbiased.

## 1 · Setup

In [ ]:
import os, time, numpy as np, torch
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", device)
if device == "cpu":
    print("WARNING: no GPU — enable Runtime > Change runtime type > GPU.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2 · Configuration

In [ ]:
# path to cohort
V2_CACHE = "..."

# experiment scale
N_SUB   = 12000
N_GEN   = 8000
B_SCALES = [0.0, 1.0]
PHASE_A_EPOCHS = (8, 5, 8)

# model dims
D_Z, D_E, D_RNN, D_COMB, H_MASK, H_DEN, N_DIFF = 32, 128, 128, 128, 256, 256, 100

# injected-mechanism design (per variable class)
BASE_B   = {"triggered": 1.0, "acuity": 0.5, "protocol": 0.2}   # value-on-recording strength
COUPLING = {"triggered": 0.7, "acuity": 0.5, "protocol": 0.3}   # how strongly x tracks severity
SEV_W    = {"triggered": 1.0, "acuity": 0.5, "protocol": 0.2}   # severity-score loadings
AR_RHO   = 0.8             # severity temporal smoothness
SEED     = 0
print("Phase A config | N_SUB", N_SUB, "| b_scales", B_SCALES, "| schedule", PHASE_A_EPOCHS)

## 3 · Model core (embedded, with `return_x` generation)

In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


# utils
def timestep_embedding(t, dim):
    """Sinusoidal embedding of diffusion step t:(B,) -> (B,dim)."""
    half = dim // 2
    freqs = torch.exp(-math.log(10000.0) * torch.arange(half, device=t.device).float() / max(half, 1))
    args = t.float()[:, None] * freqs[None]
    emb = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
    if dim % 2:
        emb = F.pad(emb, (0, 1))
    return emb


def causal_hist(m):
    """m:(B,T,V) {0,1} -> (B,T,2V): [count-before-t / T, steps-since-last-obs-before-t / T].
    Strictly causal (uses only t' < t), matching the incremental generation-time update."""
    B, T, V = m.shape
    csum = torch.cumsum(m, dim=1)
    count_excl = csum - m
    idx = torch.arange(T, device=m.device).view(1, T, 1).float().expand(B, T, V)
    obs_pos = torch.where(m > 0.5, idx, torch.full_like(m, -1.0))
    shifted = torch.cat([torch.full((B, 1, V), -1.0, device=m.device), obs_pos[:, :-1, :]], dim=1).contiguous()
    last_obs_excl = torch.cummax(shifted, dim=1).values
    dt = idx - last_obs_excl
    return torch.cat([count_excl / T, dt / T], dim=-1)


class DiffusionSchedule:
    """Standard DDPM linear-beta schedule with precomputed coefficients."""
    def __init__(self, n_steps=100, beta_start=1e-4, beta_end=2e-2):
        betas = torch.linspace(beta_start, beta_end, n_steps)
        alphas = 1.0 - betas
        abar = torch.cumprod(alphas, dim=0)
        abar_prev = torch.cat([torch.ones(1), abar[:-1]])
        self.n_steps = n_steps
        self.betas = betas
        self.alphas = alphas
        self.abar = abar
        self.sqrt_abar = torch.sqrt(abar)
        self.sqrt_one_minus_abar = torch.sqrt(1.0 - abar)
        self.sqrt_recip_alphas = torch.sqrt(1.0 / alphas)
        self.posterior_var = betas * (1.0 - abar_prev) / (1.0 - abar)

    def to(self, device):
        for k, v in list(self.__dict__.items()):
            if torch.is_tensor(v):
                setattr(self, k, v.to(device))
        return self

    def q_sample(self, x0, t, noise):
        sa = self.sqrt_abar[t].view(-1, 1, 1)
        soma = self.sqrt_one_minus_abar[t].view(-1, 1, 1)
        return sa * x0 + soma * noise


class Denoiser(nn.Module):
    """Predicts diffusion noise per timestep: eps_theta(x_noisy, cond, t). Applied vectorised over T."""
    def __init__(self, V, d_cond, d_temb=64, hidden=256):
        super().__init__()
        self.d_temb = d_temb
        self.net = nn.Sequential(
            nn.Linear(V + d_cond + d_temb, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, V),
        )

    def forward(self, x, cond, t):
        temb = timestep_embedding(t, self.d_temb)
        temb = temb[:, None, :].expand(-1, x.shape[1], -1)
        return self.net(torch.cat([x, cond, temb], dim=-1))


# model
class RecordDiff(nn.Module):
    def __init__(self, V, d_c=8, d_z=32, d_e=128, d_rnn=128, d_comb=128,
                 hidden_mask=256, hidden_den=256, d_temb=64, n_diff=100):
        super().__init__()
        self.V, self.d_c, self.d_z = V, d_c, d_z
        self.diff = DiffusionSchedule(n_diff)

        self.emb = nn.Sequential(nn.Linear(2 * V, d_e), nn.SiLU(), nn.Linear(d_e, d_e))
        self.bigru = nn.GRU(d_e, d_rnn, batch_first=True, bidirectional=True)
        self.comb_z = nn.Linear(d_z, d_comb)
        self.comb_g = nn.Linear(2 * d_rnn, d_comb)
        self.q_mu = nn.Linear(d_comb, d_z)
        self.q_ls = nn.Linear(d_comb, d_z)
        self.p0_mu = nn.Linear(d_c, d_z)
        self.p0_ls = nn.Linear(d_c, d_z)
        self.tr_body = nn.Sequential(nn.Linear(d_z + d_c, d_comb), nn.SiLU())
        self.tr_mu = nn.Linear(d_comb, d_z)
        self.tr_ls = nn.Linear(d_comb, d_z)
        self.mask_head = nn.Sequential(
            nn.Linear(d_z + d_c + 2 * V, hidden_mask), nn.SiLU(),
            nn.Linear(hidden_mask, hidden_mask), nn.SiLU(),
            nn.Linear(hidden_mask, V))
        self.d_cond = d_z + d_c + V + 2 * V
        self.denoiser = Denoiser(V, self.d_cond, d_temb, hidden_den)
        self.register_buffer("norm_mean", torch.zeros(V))
        self.register_buffer("norm_std", torch.ones(V))

    def params_encoder(self):
        mods = [self.emb, self.bigru, self.comb_z, self.comb_g, self.q_mu, self.q_ls,
                self.p0_mu, self.p0_ls, self.tr_body, self.tr_mu, self.tr_ls]
        return [p for mod in mods for p in mod.parameters()]

    def params_value(self):
        return list(self.denoiser.parameters())

    def params_mask(self):
        return list(self.mask_head.parameters())

    def set_normalizer(self, mean, std):
        self.norm_mean.data = mean.to(self.norm_mean.device)
        self.norm_std.data = std.clamp(min=1e-3).to(self.norm_std.device)

    def _prior_step(self, z_prev, c, t):
        if t == 0:
            return self.p0_mu(c), self.p0_ls(c)
        body = self.tr_body(torch.cat([z_prev, c], dim=-1))
        return z_prev + self.tr_mu(body), self.tr_ls(body)

    def infer(self, y, m, c):
        """Amortised posterior over z_{1:T}; returns Z and KL(q||p) (free-bits, mean over B,T)."""
        B, T, V = y.shape
        e = self.emb(torch.cat([y, m], dim=-1))
        g, _ = self.bigru(e)
        z_prev = torch.zeros(B, self.d_z, device=y.device)
        Zs, KLs = [], []
        for t in range(T):
            hc = 0.5 * (torch.tanh(self.comb_z(z_prev)) + self.comb_g(g[:, t, :]))
            mu_q, ls_q = self.q_mu(hc), self.q_ls(hc)
            mu_p, ls_p = self._prior_step(z_prev, c, t)
            std_q = F.softplus(ls_q) + 1e-4
            std_p = F.softplus(ls_p) + 1e-4
            z = mu_q + std_q * torch.randn_like(std_q)
            kl = (torch.log(std_p / std_q)
                  + (std_q ** 2 + (mu_q - mu_p) ** 2) / (2 * std_p ** 2) - 0.5)
            Zs.append(z); KLs.append(kl)
            z_prev = z
        Z = torch.stack(Zs, dim=1)
        KL = torch.stack(KLs, dim=1)
        return Z, KL

    def losses(self, y, m, c, free_bits=0.02):
        """y assumed already normalised; m in {0,1}; c:(B,d_c). Returns loss dict."""
        B, T, V = y.shape
        hist = causal_hist(m)
        Z, KL = self.infer(y, m, c)
        c_seq = c[:, None, :].expand(-1, T, -1)

        # Stage 1: measurement policy
        mask_logits = self.mask_head(torch.cat([Z, c_seq, hist], dim=-1))
        L_mask = F.binary_cross_entropy_with_logits(mask_logits, m)

        # Stage 2: masked conditional diffusion on observed values
        x0 = y
        tau = torch.randint(0, self.diff.n_steps, (B,), device=y.device)
        noise = torch.randn_like(x0)
        x_noisy = self.diff.q_sample(x0, tau, noise)
        cond = torch.cat([Z, c_seq, m, hist], dim=-1)
        eps_pred = self.denoiser(x_noisy, cond, tau)
        se = (eps_pred - noise) ** 2
        L_value = (se * m).sum() / m.sum().clamp(min=1.0)

        KL_fb = torch.clamp(KL, min=free_bits).sum(-1).mean()
        return {"L_value": L_value, "L_mask": L_mask, "KL": KL_fb}

    # generation
    @torch.no_grad()
    def _ddpm_sample(self, cond):
        """Reverse DDPM for a single timestep. cond:(n,d_cond) -> x:(n,V) (normalised)."""
        n = cond.shape[0]
        dev = cond.device
        x = torch.randn(n, 1, self.V, device=dev)
        cond1 = cond[:, None, :]
        for i in reversed(range(self.diff.n_steps)):
            tau = torch.full((n,), i, dtype=torch.long, device=dev)
            eps = self.denoiser(x, cond1, tau)
            mean = self.diff.sqrt_recip_alphas[i] * (
                x - self.diff.betas[i] / self.diff.sqrt_one_minus_abar[i] * eps)
            if i > 0:
                x = mean + torch.sqrt(self.diff.posterior_var[i]) * torch.randn_like(x)
            else:
                x = mean
        return x[:, 0, :]

    @torch.no_grad()
    def generate(self, n, c, T, denorm=True, return_x=False):
        """Ancestral sample of recorded reality. Returns (Y, M) [+ complete X if return_x],
        denormalised if denorm=True. X is the pre-mask complete value tensor."""
        dev = c.device
        V = self.V
        count_run = torch.zeros(n, V, device=dev)
        last_obs = -torch.ones(n, V, device=dev)
        z_prev = torch.zeros(n, self.d_z, device=dev)
        M = torch.zeros(n, T, V, device=dev)
        Y = torch.zeros(n, T, V, device=dev)
        X = torch.zeros(n, T, V, device=dev) if return_x else None
        for t in range(T):
            mu_p, ls_p = self._prior_step(z_prev, c, t)
            z = mu_p + (F.softplus(ls_p) + 1e-4) * torch.randn_like(mu_p)
            hist = torch.cat([count_run / T, (t - last_obs) / T], dim=-1)
            m_t = torch.bernoulli(torch.sigmoid(self.mask_head(torch.cat([z, c, hist], dim=-1))))
            x_t = self._ddpm_sample(torch.cat([z, c, m_t, hist], dim=-1))
            if denorm:
                x_t = x_t * self.norm_std + self.norm_mean
            M[:, t, :] = m_t
            Y[:, t, :] = m_t * x_t
            if return_x:
                X[:, t, :] = x_t
            last_obs = torch.where(m_t > 0.5, torch.full_like(last_obs, float(t)), last_obs)
            count_run = count_run + m_t
            z_prev = z
        return (Y, M, X) if return_x else (Y, M)


# data helpers
def fit_normalizer(y, m):
    """Per-variable mean/std over observed (m==1) entries. y,m:(N,T,V) tensors."""
    V = y.shape[-1]
    mean = torch.zeros(V); std = torch.ones(V)
    for v in range(V):
        vals = y[:, :, v][m[:, :, v] > 0.5]
        if vals.numel() > 10:
            mean[v] = vals.mean()
            std[v] = vals.std().clamp(min=1e-3)
    return mean, std


def normalize(y, m, mean, std):
    yn = (y - mean) / std
    return yn * m


def set_requires_grad(params, flag):
    for p in params:
        p.requires_grad_(flag)


def train_recorddiff(model, y, m, c, epochs=(8, 4, 8), batch=256, lr=1e-3,
                     beta_max=1.0, lambda_m=1.0, free_bits=0.02, val_frac=0.1,
                     clip=5.0, device="cpu", seed=0, verbose=True):
    """3-phase amortised-VI training.
        Phase 1 (representation): encoder + value diffusion + prior,  loss = L_value + beta*KL
        Phase 2 (policy):         freeze the above, train mask head,   loss = L_mask
        Phase 3 (joint):          everything,  loss = L_value + lambda_m*L_mask + beta_max*KL
    y is RAW (normalised internally via model.norm_*). Returns loss history."""
    torch.manual_seed(seed)
    model.to(device); model.diff.to(device)
    N = y.shape[0]
    perm = torch.randperm(N)
    n_val = int(N * val_frac)
    val_idx, tr_idx = perm[:n_val], perm[n_val:]
    e1, e2, e3 = epochs
    total = e1 + e2 + e3
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    mean, std = model.norm_mean.cpu(), model.norm_std.cpu()
    hist = {"phase": [], "L_value": [], "L_mask": [], "KL": [], "val_total": []}

    def run_batches(idx, train=True):
        agg = {"L_value": 0.0, "L_mask": 0.0, "KL": 0.0, "tot": 0.0, "nb": 0}
        order = idx[torch.randperm(len(idx))] if train else idx
        for s in range(0, len(order), batch):
            bi = order[s:s + batch]
            yb = normalize(y[bi], m[bi], mean, std).to(device)
            mb = m[bi].to(device)
            cb = c[bi].to(device)
            out = model.losses(yb, mb, cb, free_bits=free_bits)
            if phase == 1:
                loss = out["L_value"] + beta * out["KL"]
            elif phase == 2:
                loss = out["L_mask"]
            else:
                loss = out["L_value"] + lambda_m * out["L_mask"] + beta_max * out["KL"]
            if train:
                opt.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), clip); opt.step()
            for k in ("L_value", "L_mask", "KL"):
                agg[k] += float(out[k].detach())
            agg["tot"] += float(loss.detach()); agg["nb"] += 1
        for k in ("L_value", "L_mask", "KL", "tot"):
            agg[k] /= max(agg["nb"], 1)
        return agg

    for ep in range(total):
        if ep < e1:
            phase = 1; beta = beta_max * min(1.0, (ep + 1) / max(e1, 1))
            set_requires_grad(model.params_encoder(), True)
            set_requires_grad(model.params_value(), True)
            set_requires_grad(model.params_mask(), False)
        elif ep < e1 + e2:
            phase = 2; beta = beta_max
            set_requires_grad(model.params_encoder(), False)
            set_requires_grad(model.params_value(), False)
            set_requires_grad(model.params_mask(), True)
        else:
            phase = 3; beta = beta_max
            set_requires_grad(model.params_encoder(), True)
            set_requires_grad(model.params_value(), True)
            set_requires_grad(model.params_mask(), True)

        model.train(); tr = run_batches(tr_idx, train=True)
        model.eval()
        with torch.no_grad():
            va = run_batches(val_idx, train=False) if n_val > 0 else tr
        hist["phase"].append(phase)
        hist["L_value"].append(tr["L_value"]); hist["L_mask"].append(tr["L_mask"])
        hist["KL"].append(tr["KL"]); hist["val_total"].append(va["tot"])
        if verbose:
            print(f"ep {ep:3d} | phase {phase} | "
                  f"L_value {tr['L_value']:.4f}  L_mask {tr['L_mask']:.4f}  KL {tr['KL']:.4f} "
                  f"| val_total {va['tot']:.4f}")
    return hist


# synthetic
def make_synthetic(n=512, T=12, V=6, seed=0, device="cpu"):
    """Toy MNAR data: latent severity drives both values and (informatively) the mask."""
    g = torch.Generator().manual_seed(seed)
    sev = torch.rand(n, 1, 1, generator=g)                     # patient severity in [0,1]
    base = torch.randn(n, 1, V, generator=g)
    drift = torch.linspace(0, 1, T).view(1, T, 1) * (sev - 0.5) * 4.0
    x = base + drift + 0.3 * torch.randn(n, T, V, generator=g)  # complete values
    # MNAR mask: higher severity -> more measurement
    logit = -0.5 + 2.0 * sev + 0.6 * (x > 1.0).float() + 0.4 * torch.randn(n, T, V, generator=g)
    m = torch.bernoulli(torch.sigmoid(logit), generator=g)
    y = x * m
    c = torch.cat([sev.view(n, 1), torch.randn(n, 7, generator=g)], dim=-1)  # d_c=8
    return y.to(device), m.to(device), c.to(device)


## 4 · Load real marginals from the cohort cache

In [ ]:
d = np.load(V2_CACHE, allow_pickle=True)
m_real, y_real = d["m"].astype(np.float32), d["y"].astype(np.float32)
VAR_NAMES = list(d["var_names"]); VAR_CLASS = [str(x) for x in d["var_class"]]
c_all = torch.from_numpy(d["c"].astype(np.float32))
Nall, T, V = m_real.shape

REAL_MEAN = np.zeros(V, np.float32); REAL_STD = np.ones(V, np.float32)
for v in range(V):
    vals = y_real[:, :, v][m_real[:, :, v] > 0.5]
    if vals.size > 200:
        lo, hi = np.quantile(vals, [0.005, 0.995]); vals = np.clip(vals, lo, hi)
        REAL_MEAN[v] = vals.mean(); REAL_STD[v] = max(vals.std(), 1e-3)
REAL_RATE = m_real.mean(axis=(0, 1)).astype(np.float32)

# subsample covariates for the semi-synthetic patients
rng = np.random.default_rng(SEED)
sub = rng.choice(Nall, size=min(N_SUB, Nall), replace=False)
c_sub = c_all[sub]
N_SUB = c_sub.shape[0]; D_C = c_sub.shape[1]
print(f"real marginals loaded | V={V} T={T} | N_SUB={N_SUB} d_c={D_C}")
print(f"real missing-rate range: [{REAL_RATE.min():.3f}, {REAL_RATE.max():.3f}]")

## 5 · Semi-synthetic generator, severity, injection, and recovery estimator


In [ ]:

BASE_B_V  = np.array([BASE_B[c]   for c in VAR_CLASS], np.float32)
COUPLE_V  = np.array([COUPLING[c] for c in VAR_CLASS], np.float32)
W_V       = np.array([SEV_W[c]    for c in VAR_CLASS], np.float32)
S_SCALE   = float(np.sqrt((W_V ** 2).sum()))

def severity(x):
    z = (x - REAL_MEAN[None, None, :]) / REAL_STD[None, None, :]
    return (z * W_V[None, None, :]).sum(-1) / S_SCALE

def _calibrate_intercept(b_v, s, target_rate):
    lo, hi = -20.0, 20.0
    for _ in range(60):
        a = 0.5 * (lo + hi)
        r = (1.0 / (1.0 + np.exp(-(a + b_v * s)))).mean()
        if r < target_rate: lo = a
        else: hi = a
    return 0.5 * (lo + hi)

def make_semisynth(b_scale, seed=0):
    g = np.random.default_rng(seed)
    u = np.zeros((N_SUB, T), np.float32); u[:, 0] = g.standard_normal(N_SUB)
    for t in range(1, T):
        u[:, t] = AR_RHO * u[:, t - 1] + np.sqrt(1 - AR_RHO ** 2) * g.standard_normal(N_SUB)
    eps = g.standard_normal((N_SUB, T, V)).astype(np.float32)
    z = COUPLE_V[None, None, :] * u[:, :, None] + np.sqrt(1 - COUPLE_V ** 2)[None, None, :] * eps
    x = REAL_MEAN[None, None, :] + REAL_STD[None, None, :] * z
    s = severity(x)                                            # (N,T)
    b_true = (b_scale * BASE_B_V).astype(np.float32)
    a = np.array([_calibrate_intercept(b_true[v], s, REAL_RATE[v]) for v in range(V)], np.float32)
    p = 1.0 / (1.0 + np.exp(-(a[None, None, :] + b_true[None, None, :] * s[:, :, None])))
    m = (g.random((N_SUB, T, V)) < p).astype(np.float32)
    return dict(x=x, y=(x * m).astype(np.float32), m=m, s=s, b_true=b_true, a=a)

def fit_b(m, s):
    sflat = s.reshape(-1, 1); V_ = m.shape[-1]; out = np.zeros(V_, np.float32)
    for v in range(V_):
        yv = m[:, :, v].ravel()
        if yv.min() == yv.max():
            out[v] = 0.0; continue
        lr = LogisticRegression(C=10.0, max_iter=300); lr.fit(sflat, yv)
        out[v] = lr.coef_[0, 0]
    return out

_D = make_semisynth(1.0, seed=SEED)
_bh = fit_b(_D["m"], _D["s"])
print(f"estimator sanity (b_scale=1): corr(b_hat_inj, b_true) = "
      f"{np.corrcoef(_bh, _D['b_true'])[0,1]:.3f}, MAE = {np.abs(_bh-_D['b_true']).mean():.3f}")

## 6 · Run the experiment: inject → train → recover (per `b_scale`)


In [ ]:
results = {}
for b_scale in B_SCALES:
    print(f"\n{'='*60}\n b_scale = {b_scale}\n{'='*60}")
    D = make_semisynth(b_scale, seed=SEED)
    yss = torch.from_numpy(D["y"]); mss = torch.from_numpy(D["m"])

    model = RecordDiff(V=V, d_c=D_C, d_z=D_Z, d_e=D_E, d_rnn=D_RNN, d_comb=D_COMB,
                       hidden_mask=H_MASK, hidden_den=H_DEN, d_temb=64, n_diff=N_DIFF).to(device)
    model.diff.to(device)
    mean_ss, std_ss = fit_normalizer(yss, mss)
    model.set_normalizer(mean_ss, std_ss)
    t0 = time.time()
    train_recorddiff(model, yss, mss, c_sub, epochs=PHASE_A_EPOCHS, batch=256, lr=1e-3,
                     beta_max=1.0, lambda_m=1.0, free_bits=0.02, device=device, seed=SEED,
                     verbose=False)
    print(f"  trained in {(time.time()-t0)/60:.1f} min")

    model.eval(); model.diff.to(device)
    gi = torch.randint(0, N_SUB, (N_GEN,))
    with torch.no_grad():
        _, Mg, Xg = model.generate(N_GEN, c_sub[gi].to(device), T, return_x=True)
    Mg, Xg = Mg.cpu().numpy(), Xg.cpu().numpy()

    b_true    = D["b_true"]
    b_hat_inj = fit_b(D["m"], D["s"])
    b_hat_gen = fit_b(Mg, severity(Xg))
    results[b_scale] = dict(b_true=b_true, b_hat_inj=b_hat_inj, b_hat_gen=b_hat_gen,
                            synth_rate=Mg.mean(0).mean(0))
    mae = np.abs(b_hat_gen - b_true).mean()
    r = np.corrcoef(b_hat_gen, b_true)[0, 1] if b_true.std() > 1e-6 else float("nan")
    print(f"  recovery: MAE(b_hat_gen, b_true) = {mae:.3f} | corr = {r:.3f} | "
          f"max|b_hat_gen| = {np.abs(b_hat_gen).max():.3f}")

## 7 · Results

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 5))
col = {"protocol": "tab:blue", "acuity": "tab:orange", "triggered": "tab:red"}
cols = [col[c] for c in VAR_CLASS]

full = 1.0 if 1.0 in results else max(results)
R = results[full]
ax[0].scatter(R["b_true"], R["b_hat_gen"], c=cols, s=30)
lim = [min(R["b_true"].min(), R["b_hat_gen"].min()) - 0.1,
       max(R["b_true"].max(), R["b_hat_gen"].max()) + 0.1]
ax[0].plot(lim, lim, "k--", lw=1)
ax[0].set_xlabel("b_true (injected)"); ax[0].set_ylabel("b_hat_gen (recovered from model)")
ax[0].set_title(f"recovery from model samples (b_scale={full})")

ax[1].scatter(R["b_true"], R["b_hat_inj"], c=cols, s=30); ax[1].plot(lim, lim, "k--", lw=1)
ax[1].set_xlabel("b_true"); ax[1].set_ylabel("b_hat_inj (on injected data)")
ax[1].set_title("estimator sanity check")

labels = ["protocol", "acuity", "triggered"]
xpos = np.arange(len(labels)); wbar = 0.35
for j, bs in enumerate(sorted(results)):
    means = [np.mean([results[bs]["b_hat_gen"][i] for i in range(V) if VAR_CLASS[i] == cl])
             for cl in labels]
    ax[2].bar(xpos + (j - 0.5) * wbar, means, wbar, label=f"b_scale={bs}")
ax[2].set_xticks(xpos); ax[2].set_xticklabels(labels); ax[2].axhline(0, c="gray", lw=0.8)
ax[2].set_ylabel("mean recovered b_hat_gen"); ax[2].set_title("null vs full, by class"); ax[2].legend()
plt.tight_layout(); plt.show()

print(f"{'b_scale':>8s} | {'MAE(gen,true)':>13s} {'corr(gen,true)':>14s} {'max|b_hat_gen|':>14s}")
print("-" * 56)
for bs in sorted(results):
    R = results[bs]
    r = np.corrcoef(R["b_hat_gen"], R["b_true"])[0, 1] if R["b_true"].std() > 1e-6 else float("nan")
    print(f"{bs:8.1f} | {np.abs(R['b_hat_gen']-R['b_true']).mean():13.3f} "
          f"{r:14.3f} {np.abs(R['b_hat_gen']).max():14.3f}")